# Catálogo do AWS Glue no AWS Glue 6.0

Neste exemplo, que roda no runtime **AWS Glue 6.0**, usamos o **Glue Data Catalog** de quatro formas: `getSink` com `enableUpdateCatalog`, `from_catalog`, `spark.sql` e uma célula **Spark SQL**. Criamos a tabela `silver.glue_orders` (Parquet, particionada por `regiao`), lemos de volta e acrescentamos dados com `write_dynamic_frame.from_catalog`. Todas as células podem ser reexecutadas.

## 1. Sessão

O runtime **AWS Glue 6.0** já configura o catálogo e o acesso ao `s3`; só precisamos criar o `GlueContext` e obter a sessão a partir dele. O database `silver` já existe no catálogo.

In [ ]:
from awsglue.context import GlueContext
from awsglue.dynamicframe import DynamicFrame
from pyspark.context import SparkContext

glueContext = GlueContext(SparkContext.getOrCreate())
spark = glueContext.spark_session

## 2. Preparar os dados

Criamos dois DataFrames pequenos: a carga inicial e um segundo lote, que acrescentaremos depois. Também removemos os arquivos que uma execução anterior deixou em `silver/glue_orders_files`; assim a sequência sempre termina com as mesmas 5 linhas.

In [ ]:
schema = "order_id INT, cliente STRING, valor DOUBLE, regiao STRING"

pedidos_iniciais = spark.createDataFrame(
    [(1, "Ana", 120.50, "sudeste"), (2, "Bruno", 80.00, "sul"), (3, "Carla", 45.90, "sudeste")],
    schema,
)
pedidos_novos = spark.createDataFrame(
    [(4, "Diego", 300.00, "sul"), (5, "Elisa", 150.75, "nordeste")],
    schema,
)

destino = "s3://notislake/silver/glue_orders_files"
caminho = spark._jvm.org.apache.hadoop.fs.Path(destino)
fs = caminho.getFileSystem(spark._jsc.hadoopConfiguration())
if fs.exists(caminho):
    fs.delete(caminho, True)

## 3. Gravar no catálogo com getSink

`getSink` grava os arquivos em `s3://notislake/silver/glue_orders_files` e, com `enableUpdateCatalog=True` e `updateBehavior="UPDATE_IN_DATABASE"`, cria ou atualiza a tabela `silver.glue_orders` e registra as partições no catálogo. Reexecutar mantém a mesma `location` e apenas atualiza a tabela.

In [ ]:
iniciais_dyf = DynamicFrame.fromDF(pedidos_iniciais, glueContext, "pedidos_iniciais")

sink = glueContext.getSink(
    path=destino,
    connection_type="s3",
    updateBehavior="UPDATE_IN_DATABASE",
    partitionKeys=["regiao"],
    enableUpdateCatalog=True,
)
sink.setCatalogInfo(catalogDatabase="silver", catalogTableName="glue_orders")
sink.setFormat("glueparquet")
sink.writeFrame(iniciais_dyf)

## 4. Ler com from_catalog

`from_catalog` lê a tabela pelo nome no catálogo, sem informar caminho nem schema. Com `push_down_predicate` lemos só as partições que atendem ao filtro.

In [ ]:
pedidos_dyf = glueContext.create_dynamic_frame.from_catalog(database="silver", table_name="glue_orders")
echo(pedidos_dyf.toDF().orderBy("order_id"), n=20)

In [ ]:
sul_dyf = glueContext.create_dynamic_frame.from_catalog(
    database="silver",
    table_name="glue_orders",
    push_down_predicate="regiao = 'sul'",
)
echo(sul_dyf.toDF().orderBy("order_id"), n=20)

## 5. Consultar com spark.sql

A tabela do catálogo também fica visível para o Spark. Em PySpark usamos `spark.sql`:

In [ ]:
echo(spark.sql("""
    SELECT regiao, COUNT(*) AS pedidos, ROUND(SUM(valor), 2) AS total
    FROM silver.glue_orders
    GROUP BY regiao
    ORDER BY regiao
"""), n=20)

E em uma célula Spark SQL:

In [ ]:
SELECT order_id, cliente, valor, regiao
FROM silver.glue_orders
ORDER BY order_id

## 6. Acrescentar com write_dynamic_frame

`write_dynamic_frame.from_catalog` grava no `location` e no formato já registrados na tabela e acrescenta o segundo lote, que inclui a nova partição `nordeste`.

In [ ]:
novos_dyf = DynamicFrame.fromDF(pedidos_novos, glueContext, "pedidos_novos")

glueContext.write_dynamic_frame.from_catalog(
    frame=novos_dyf,
    database="silver",
    table_name="glue_orders",
)

## 7. Consultar o resultado

O estado final tem 5 linhas, o mesmo a cada reexecução. Com `from_catalog`:

In [ ]:
resultado_dyf = glueContext.create_dynamic_frame.from_catalog(database="silver", table_name="glue_orders")
echo(resultado_dyf.toDF().orderBy("order_id"), n=20)

E em Spark SQL:

In [ ]:
SELECT regiao, COUNT(*) AS pedidos, ROUND(SUM(valor), 2) AS total
FROM silver.glue_orders
GROUP BY regiao
ORDER BY regiao